# ЛР1 · 03b. Baseline: LightGBM, обученный на наших данных

Сильный «классический» baseline (как GBDT в PVCGN и LightGBM в работе Мартыненко и Ваколюка):
одна глобальная модель на все 424 станции. Прогноз на сутки вперёд из точки 00:00 — поэтому все признаки
известны на момент прогноза: лаги не короче 24 ч, календарь, режимы, события на площадках, признаки станции,
погода (фактическая — допущение «идеального прогноза погоды», PROTOCOL.md).

- Обучение: train с 01.01.2025 (оба структурных сдвига уже позади; ~15 месяцев), закрытые часы исключены.
- Ранняя остановка: по val. Оценка: те же 600 окон теста, что у всех моделей.
- Целевая функция `poisson` — естественна для счётных данных и учитывает разный масштаб станций.

**Kaggle:** Inputs — датасет и вывод `lab-00-setup`; Accelerator — **None** (CPU); Internet не нужен.

In [ ]:
import glob, json, sys, time, gc
from pathlib import Path
import numpy as np, pandas as pd, lightgbm as lgb
hits = glob.glob("/kaggle/input/**/labkit.py", recursive=True) or glob.glob("labkit.py")
LAB00 = Path(hits[0]).parent
sys.path.insert(0, str(LAB00))
import labkit as lk
print("lightgbm", lgb.__version__)

d = lk.load()
sub = json.load(open(LAB00 / "subset.json", encoding="utf-8"))
stations = sub["stations"]
test_orig = [pd.Timestamp(x) for x in sub["test_origins"]]
val_orig = [pd.Timestamp(x) for x in sub["val_origins"]]
assert stations == lk.make_subset(d)
tmax = d.train_max()
W_test = lk.make_windows(d, stations, test_orig, context=336)
W_val = lk.make_windows(d, stations, val_orig, context=336)

## 1. Признаки (широкие массивы час × станция → длинная таблица)

In [ ]:
idx = d.y.index; cols = list(d.y.columns); S = len(cols)
Y = d.y.to_numpy(dtype="float32")
C = d.closed.to_numpy()
Ym = np.where(C, np.nan, Y)
P0 = idx.get_loc(pd.Timestamp("2025-01-01 00:00"))
T = len(idx) - P0

def lag(k):
    return Ym[P0 - k:len(idx) - k]                     # [T, S]

F = {}
for k in [24, 48, 72, 168, 336, 504, 672]:
    F[f"lag{k}"] = lag(k)
F["lag_week_mean4"] = np.nanmean(np.stack([lag(168), lag(336), lag(504), lag(672)]), axis=0)
F["lag_week_std4"] = np.nanstd(np.stack([lag(168), lag(336), lag(504), lag(672)]), axis=0)
# сумма вчерашнего дня (известна в 00:00): для часа h это сумма часов [t-h-24, t-h)
hour = idx.hour.to_numpy()[P0:]
day_sum = pd.DataFrame(Ym).groupby(idx.normalize()).transform("sum").to_numpy()   # сумма своего дня
F["yesterday_total"] = np.vstack([day_sum[P0 - 24:len(idx) - 24]])

# ковариаты по времени: календарь, режимы, события, погода
COV_COLS = [c for c in d.cov.columns if c not in ("dayofyear", "week_of_year")]
cov = d.cov[COV_COLS].to_numpy(dtype="float32")[P0:]

# станции
st = d.static.loc[cols]
ST_COLS = ["scale_weekday", "commute_index", "cluster", "in_cbd", "n_lines", "od_out_flow", "od_in_flow", "latitude", "longitude"]
stv = st[ST_COLS].astype("float32").to_numpy()

# события на площадках
ev = pd.read_parquet(d.root / "events_station_hour.parquet")
ev["station_complex_id"] = ev["station_complex_id"].astype(str)
EV = {}
for c in ["ev_today", "ev_active", "ev_hours_from_start"]:
    wide = ev.pivot_table(index="transit_timestamp", columns="station_complex_id", values=c, aggfunc="first")
    wide = wide.reindex(index=idx, columns=cols)
    EV[c] = wide.to_numpy(dtype="float32")[P0:]
EV["ev_today"] = np.nan_to_num(EV["ev_today"]); EV["ev_active"] = np.nan_to_num(EV["ev_active"])

names = list(F) + list(EV) + COV_COLS + ST_COLS + ["station"]
X = np.empty((T * S, len(names)), dtype="float32")
j = 0
for a in list(F.values()) + list(EV.values()):
    X[:, j] = a.reshape(-1); j += 1
for k in range(cov.shape[1]):
    X[:, j] = np.repeat(cov[:, k], S); j += 1
for k in range(stv.shape[1]):
    X[:, j] = np.tile(stv[:, k], T); j += 1
X[:, j] = np.tile(np.arange(S), T)
y = Y[P0:].reshape(-1)
open_ = ~C[P0:].reshape(-1)
ts = np.repeat(idx[P0:].to_numpy(), S)
print(f"Таблица: {X.shape[0]:,} строк × {X.shape[1]} признаков, {X.nbytes / 1e9:.1f} ГБ")
del F, day_sum; gc.collect()

## 2. Обучение

In [ ]:
tr_end = d.part("train")[1]; va0, va1 = d.part("val")
m_tr = open_ & (ts <= np.datetime64(tr_end))
m_va = open_ & (ts >= np.datetime64(va0)) & (ts <= np.datetime64(va1))
dtr = lgb.Dataset(X[m_tr], y[m_tr], feature_name=names, categorical_feature=["station", "cluster"], free_raw_data=True)
dva = lgb.Dataset(X[m_va], y[m_va], reference=dtr)
params = dict(objective="poisson", learning_rate=0.05, num_leaves=255, min_data_in_leaf=200,
              feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
              num_threads=-1, seed=lk.SEED, verbose=-1)
t0 = time.time()
booster = lgb.train(params, dtr, num_boost_round=1500, valid_sets=[dva], valid_names=["val"],
                    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)])
train_min = (time.time() - t0) / 60
print(f"Обучение: {train_min:.1f} мин, деревьев {booster.best_iteration}")
imp = pd.Series(booster.feature_importance("gain"), index=names).sort_values(ascending=False)
imp.to_csv(lk.results_dir() / "lightgbm_feature_importance.csv")
print("Топ-15 признаков по gain:"); print((imp / imp.sum()).head(15).round(3).to_string())

## 3. Прогноз на тех же окнах, что у всех моделей

In [ ]:
pos_of = {t: i for i, t in enumerate(idx[P0:])}
col_of = {s: k for k, s in enumerate(cols)}

def predict_windows(w):
    rows = []
    for s, o in zip(w.meta["station"], w.meta["origin"]):
        p = pos_of[o]
        rows.append([(p + h) * S + col_of[s] for h in range(lk.HORIZON)])
    rows = np.array(rows)
    t = time.perf_counter()
    pred = booster.predict(X[rows.reshape(-1)], num_iteration=booster.best_iteration).reshape(len(w), lk.HORIZON)
    ms = (time.perf_counter() - t) * 1000 / len(w)
    return pred, ms

for name, w in [("val", W_val), ("test", W_test)]:
    pred, ms = predict_windows(w)
    pred = np.clip(pred, 0, None)
    valid = lk.check_valid(list(pred), w, tmax)
    point = lk.apply_fallback(list(pred), valid, w)
    row = lk.save_run("lightgbm", f"trained_{name}", w, point, valid, None,
                      {"ms_p50": ms, "ms_p95": ms, "total_s": train_min * 60},
                      {"weights": "-", "context": "лаги до 672 ч", "train_minutes": round(train_min, 1),
                       "n_trees": booster.best_iteration, "train_rows": int(m_tr.sum())})
    print(f"LightGBM {name}: WAPE={row['wape']:.4f} [{row['wape_ci_lo']:.4f}; {row['wape_ci_hi']:.4f}], "
          f"MASE={row['mase']:.3f}, невалидных {row['invalid_share']:.1%}")
booster.save_model(str(lk.results_dir() / "lightgbm_model.txt"), num_iteration=booster.best_iteration)